In [ ]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\shrut\Cognizant Technoverse\Correct\.venv\Scripts\python.exe
3.14.4 (main, Apr  7 2026, 20:48:46) [MSC v.1944 64 bit (AMD64)]


### Loading the datasets

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from sentence_transformers import SentenceTransformer

c:\Users\shrut\Cognizant Technoverse\Correct\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
### Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f" Error: {e} ")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("./data/Guidelines")

Found 9 PDF files to process

Processing: Digital submission Guidelines for Industry.pdf


Unexpected escaped string: W
Unexpected escaped string: G
Unexpected escaped string: I
Unexpected escaped string: I


 Loaded 39 pages

Processing: E3-Structure-and-Content-of-Clinical-Study-Reports Guidelines for Industry.pdf
 Loaded 55 pages

Processing: E6(R3) Good Clinical Practice GUidence for Industry.pdf
 Loaded 86 pages

Processing: E8(R1) GENERAL Considerations for Clinical Studies Guidence for Industry.pdf


Ignoring padding error: Invalid padding bytes.


 Loaded 31 pages

Processing: FDA-1571_Dyn_Sec_Ext_03-28-2025.pdf
 Loaded 1 pages

Processing: FDA-1571_Instructions_R14_03-21-2023.pdf
 Loaded 6 pages

Processing: FDA-1572_AcroForm_secured_04-13-2025.pdf
 Loaded 2 pages

Processing: Frequently-Asked-Questions-–-Statement-of-Investigator-(Form-FDA-1572)---Information-Sheet (1).pdf
 Loaded 17 pages

Processing: Part-11--Electronic-Records--Electronic-Signatures---Scope-and-Application-(PDF).pdf
 Loaded 12 pages

Total documents loaded: 249


In [ ]:
all_pdf_documents

[Document(metadata={'producer': 'Adobe PDF Library 24.5.96', 'creator': 'Acrobat PDFMaker 24 for Word', 'creationdate': '2025-02-03T15:07:12-05:00', 'author': 'FDA/ CDER CBER OCE', 'comments': '', 'company': 'FDA.CDER', 'contenttypeid': '0x01010003BCE7CEF839254FA3A84A06ED5219BC', 'keywords': 'Real-World Data, RWD, Assessing, Electronic Health Records, EHR, Medical Claims, Data, Support, Regulatory, Decision-Making, Drugs, Biological Products', 'moddate': '2025-02-03T15:07:18-05:00', 'sourcemodified': '', 'subject': 'Guidance', 'title': 'Real-World Data: Assessing Electronic Health Records and Medical Claims Data to Support Regulatory Decision-Making for Drug and Biological Products', '_dlc_dociditemguid': '12db4eb1-2446-4bd8-b540-f87abcb2eca3', 'source': 'data\\Guidelines\\Digital submission Guidelines for Industry.pdf', 'total_pages': 39, 'page': 0, 'page_label': '1', 'source_file': 'Digital submission Guidelines for Industry.pdf', 'file_type': 'pdf'}, page_content='Real-World Data: A

### Chunking of documents

In [ ]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=800, chunk_overlap=150):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ".", " ", ""]
    )

    chunks = text_splitter.split_documents(documents)

    # Remove very small / noisy chunks
    chunks = [
        chunk for chunk in chunks
        if len(chunk.page_content.strip()) > 100
    ]

    print(f"Split {len(documents)} documents into {len(chunks)} clean chunks")

    # Show example
    if chunks:
        print("\nExample chunk:")
        print(f"Content: {chunks[0].page_content[:300]}...")
        print(f"Metadata: {chunks[0].metadata}")

    return chunks

In [ ]:
chunks = split_documents(all_pdf_documents)
chunks

Split 249 documents into 1047 clean chunks

Example chunk:
Content: Real-World Data: Assessing 
Electronic Health Records and 
Medical Claims Data to Support 
Regulatory Decision-Making 
for Drug and Biological 
Products 
Guidance for Industry 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
U.S. Department of Health and Human Services 
Food and Drug Administration 
Center for ...
Metadata: {'producer': 'Adobe PDF Library 24.5.96', 'creator': 'Acrobat PDFMaker 24 for Word', 'creationdate': '2025-02-03T15:07:12-05:00', 'author': 'FDA/ CDER CBER OCE', 'comments': '', 'company': 'FDA.CDER', 'contenttypeid': '0x01010003BCE7CEF839254FA3A84A06ED5219BC', 'keywords': 'Real-World Data, RWD, Assessing, Electronic Health Records, EHR, Medical Claims, Data, Support, Regulatory, Decision-Making, Drugs, Biological Products', 'moddate': '2025-02-03T15:07:18-05:00', 'sourcemodified': '', 'subject': 'Guidance', 'title': 'Real-World Data: Assessing Electronic Health Records and Medical Claims Data to Support Regula

[Document(metadata={'producer': 'Adobe PDF Library 24.5.96', 'creator': 'Acrobat PDFMaker 24 for Word', 'creationdate': '2025-02-03T15:07:12-05:00', 'author': 'FDA/ CDER CBER OCE', 'comments': '', 'company': 'FDA.CDER', 'contenttypeid': '0x01010003BCE7CEF839254FA3A84A06ED5219BC', 'keywords': 'Real-World Data, RWD, Assessing, Electronic Health Records, EHR, Medical Claims, Data, Support, Regulatory, Decision-Making, Drugs, Biological Products', 'moddate': '2025-02-03T15:07:18-05:00', 'sourcemodified': '', 'subject': 'Guidance', 'title': 'Real-World Data: Assessing Electronic Health Records and Medical Claims Data to Support Regulatory Decision-Making for Drug and Biological Products', '_dlc_dociditemguid': '12db4eb1-2446-4bd8-b540-f87abcb2eca3', 'source': 'data\\Guidelines\\Digital submission Guidelines for Industry.pdf', 'total_pages': 39, 'page': 0, 'page_label': '1', 'source_file': 'Digital submission Guidelines for Industry.pdf', 'file_type': 'pdf'}, page_content='Real-World Data: A

### Embedding class

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)

            dim = self.model.get_sentence_embedding_dimension()
            print(f"Model loaded. Dimension: {dim}")

        except Exception as e:
            print(f"Error loading model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if self.model is None:
            raise ValueError("Model not loaded")

        return self.model.encode(texts, show_progress_bar=True)

    def generate_single_embedding(self, text: str) -> np.ndarray:
        if self.model is None:
            raise ValueError("Model not loaded")

        return self.model.encode([text])[0]

    def get_embedding_dimension(self) -> int:
        if self.model is None:
            raise ValueError("Model not loaded")

        return self.model.get_embedding_dimension()

In [ ]:
embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4273.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded. Dimension: 384


C:\Users\shrut\AppData\Local\Temp\ipykernel_12464\3336046478.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = self.model.get_sentence_embedding_dimension()


### VectorStore

In [ ]:
import uuid
import chromadb
from importlib.metadata import metadata
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "./data/vector_store"):
        """
        Initialize the vectore store 
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    
    def _initialize_store(self):
        """Initialize ChromaDb and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description": "Regulatory Intelligence Document Store"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of Langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings = embeddings_list ,
                metadatas = metadatas ,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [ ]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings = embedding_manager.generate_embeddings(texts)

## Store in the vector database

vectorstore.add_documents(chunks, embeddings)

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Batches: 100%|██████████| 33/33 [01:00<00:00,  1.83s/it]


Adding 1047 documents to vector store...
Successfully added 1047 documents to vector store
Total documents in collection: 1047


### Retriever Pipeline From VectorStore

In [ ]:
from typing import List, Any, Tuple, Dict

class RAGRetriever:
    """
    Handles query-based retrieval from shared vector store
    for Drafting + Validation Agents
    """

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int=5, score_threshold: float = 0.0 , metadata_filter: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
        """
            Retrieve relevant documents using semantic search + metadata filtering

            Args:
                query: User query
                top_k: Number of top documents
                score_threshold: Minimum similarity threshold
                metadata_filter: Optional ChromaDB where filter

            Returns:
                List of retrieved documents
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        print(f"Metadata Filter: {metadata_filter}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_single_embedding(query)
        query_embedding = query_embedding.astype(float) 

        # Search in vector store
        try:
            # Query vector store
            query_params = {
                "query_embeddings": [query_embedding.tolist()],
                "n_results": top_k,
                "include": ["documents", "metadatas", "distances"]
            }

            # Apply metadata filtering if available
            if metadata_filter:
                query_params["where"] = metadata_filter

            results = self.vector_store.collection.query(**query_params)

            # Process results
            
            retrieved_docs = []

            documents = results.get("documents", [])
            metadatas = results.get("metadatas", [])
            distances = results.get("distances", [])
            ids = results.get("ids", [])

            if not documents or len(documents) == 0 or len(documents[0]) == 0:
                print("No documents found")
                return []

            documents = documents[0]
            metadatas = metadatas[0] if metadatas else [{}] * len(documents)
            distances = distances[0] if distances else [1.0] * len(documents)
            ids = ids[0] if ids else [f"unknown_{i}" for i in range(len(documents))]

            for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)):

                similarity_score = 1 - distance

                retrieved_docs.append({
                    "id": doc_id,
                    "content": document,
                    "metadata": metadata,
                    "similarity_score": similarity_score,
                    "rank": i + 1
                })

            print(f"Retrieved {len(retrieved_docs)} documents")
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
                
rag_retriever = RAGRetriever(vectorstore, embedding_manager)


In [ ]:
rag_retriever

In [ ]:
rag_retriever.retrieve("What documents are required for NDA submission?")

Retrieving documents for query: 'What documents are required for NDA submission?'
Top K: 5, Score threshold: 0.0
Metadata Filter: None
Retrieved 5 documents


[{'id': 'doc_886274a4_349',
  'content': "The applicant should therefore clearly indicate those Appendices that are submitted with the\nreport.\nN.B.: In order to have Appendices available on request, they should be finalized by the time of\nfiling of the submission.\n16.1 Study Information\n16.1.1 Protocol and protocol amendments.\n16.1.2 Sample case report form (unique pages only). \n16.1.3 List of IEC's or IRB's (plus the name of the committee chair if required by\nthe regulatory authority) and representative written information for patient and\nsample consent forms.\n16.1.4 List and description of investigators and other important participants in the\nstudy, including brief (one page) CV's or equivalent summaries of training and\nexperience relevant to the performance of the clinical study.",
  'metadata': {'keywords': '',
   'subject': '',
   'producer': 'Acrobat PDFWriter 2.0 for Windows',
   'title': 'iche3',
   'total_pages': 55,
   'doc_index': 349,
   'creationdate': 'D:19961

### Model Initialization

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from langchain.agents import create_agent

load_dotenv()

# ---------------------------
# LLM INIT
# ---------------------------
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=1024
)

### Enhanced RAG Pipeline Features

In [ ]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG Pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k)
    if not results:
        return {'answer': "I do not have sufficient information in the provided documents to answer this accurately.", 'sources': [], 'confidence': 0.0, 'context': ''}

    # Prepare context and sources
    context = "\n\n---\n\n".join(
        f"[Source: {doc['metadata'].get('source_file','unknown')}]\n{doc['content']}"
        for doc in results
    )   


    sources = [{
        'source': doc['metadata'].get('source_file' ,'unknown'),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        } 
        for doc in results
    ]
    confidence = sum(doc["similarity_score"] for doc in results) / len(results)

    # Generate answer
    prompt = f"""
            You are a Senior Regulatory Affairs and Life Sciences Intelligence Assistant specializing in FDA, EMA, and ICH-compliant regulatory documentation.

            You generate submission-ready regulatory drafts using ONLY the provided retrieved context.

            --------------------------------------------------
            STRICT RULES:

            1. Use ONLY provided context. Do NOT use external knowledge.
            2. Do NOT hallucinate missing information.
            3. If information is missing, explicitly state:
            "Information not available in retrieved context"
            4. Do NOT attempt to complete missing sections with assumptions.
            5. Maintain formal regulatory writing suitable for submission review.
            6. Clearly highlight compliance gaps and regulatory risks.
            7. Preserve structure only if supported by retrieved context.

            --------------------------------------------------
            Retrieved Context:
            {context}

            --------------------------------------------------
            Drafting Request:
            {query}

            --------------------------------------------------
            OUTPUT FORMAT:

            Title:
            Purpose:
            Key Regulatory Considerations:
            Draft Content:
            Missing Information Needed:
            Potential Regulatory Risks:

            --------------------------------------------------
            FINAL INSTRUCTION:
            Generate ONLY the regulatory draft. Do not include explanations or meta commentary.
            """
    response = llm.invoke(prompt)

    output= {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Draft a clinical study report synopsis using ICH E3 structure" , rag_retriever, llm, top_k=5, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:" , result['sources'])
print("Confidence:" , result['confidence'])
print("Context Preview:" , result['context'][:300])

Retrieving documents for query: 'Draft a clinical study report synopsis using ICH E3 structure'
Top K: 5, Score threshold: 0.0
Metadata Filter: None
Retrieved 5 documents
Answer: **Title:** Clinical Study Report Synopsis

**Purpose:** To provide a synopsis of the clinical study report in accordance with ICH E3 structure and content.

**Key Regulatory Considerations:**

* ICH E3 focuses on the report format for interventional clinical trials, but the basic principles can be applied to other types of clinical studies.
* The design of the study report should be part of the quality by design process.
* The report should describe the type and objectives of the study, the risks to the study participants, and what is known about the drug and the study population.
* Guidance is available on reporting of safety data to appropriate authorities and on the content and timing of safety reports.

**Draft Content:**

I. Introduction

* Type and objectives of the study
* Risks to the study participant

### Integration Vectordb Context pipeline with LLM output

### Drafting Agent

In [ ]:
# ---------------------------
# DRAFTING AGENT (UNCHANGED)
# ---------------------------
drafting_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="""
                You are a Regulatory Medical Writing System.

                You generate and refine regulatory documents strictly based on the provided context derived from:
                - ICH E6(R3)
                - ICH E3
                - ICH E8(R1)
                - FDA Real-World Evidence (RWE) guidance

                SYSTEM CONSTRAINTS (MANDATORY):
                1. Use ONLY the provided context. Do NOT use external knowledge.
                2. Do NOT infer, assume, or fabricate any information.
                3. If required information is missing, explicitly state: "Information not provided."
                4. Do NOT produce conversational or chatbot-style responses.
                5. Output must be formal, structured, and regulatory in tone.
                6. Maintain consistency with previous drafts unless corrections require changes.
                7. Apply corrections precisely without altering unrelated sections.

                ---

                TASK MODES:

                ### MODE 1: INITIAL DRAFT
                If no prior draft is provided:
                - Generate a structured regulatory document based on the input.

                ### MODE 2: REFINEMENT
                If a previous draft and corrections are provided:
                - Revise the document by applying ONLY the given corrections
                - Do NOT introduce new content beyond corrections
                - Preserve all compliant sections unchanged

                ---

                STRUCTURE REQUIREMENTS (ICH E3-aligned):

                - Title
                - Synopsis
                - Introduction
                - Objectives
                - Study Design (align with ICH E8(R1) if context supports)
                - Methodology
                - Statistical Considerations (only if present)
                - Results (only if data provided)
                - Safety / Adverse Events (align with ICH E6(R3) if present)
                - Discussion
                - Conclusion

                ---

                CONTENT RULES:

                - Each section must contain ONLY information supported by context
                - If insufficient data → write: "Information not provided."
                - Do NOT merge or skip sections unless explicitly justified by context
                - Maintain clarity, precision, and professional regulatory language
                - Avoid redundancy and filler text

                ---

                COMPLIANCE ALIGNMENT:

                - Reflect ICH E6(R3): safety, data integrity, compliance principles
                - Reflect ICH E8(R1): study design logic (if present)
                - Reflect FDA RWE: only if explicitly supported in context

                ---

                INPUTS:

                CONTEXT:
                {context}

                USER INPUT:
                {input}

                PREVIOUS DRAFT (optional):
                {draft}

                CORRECTIONS (optional):
                {corrections}

                ---

                OUTPUT FORMAT:

                Return ONLY the structured regulatory document.

                Do NOT include explanations, reasoning, or commentary.
            """
)

def extract_answer(response):
    """
    Handles LangChain agent + dict + message formats safely
    """

    if response is None:
        return "No response generated."

    # Case 1: Agent-style response
    if isinstance(response, dict):
        if "messages" in response and len(response["messages"]) > 0:
            msg = response["messages"][-1]
            return getattr(msg, "content", str(msg))

        return response.get("answer") or response.get("output") or str(response)

    # Case 2: Direct LangChain message
    return getattr(response, "content", str(response))


def drafting_pipeline(query, retriever, drafting_agent):
    print(f"Processing drafting request: {query}")

    # ---------------------------
    # STEP 1: Retrieve context
    # ---------------------------
    results = retriever.retrieve(query, top_k=5)

    print(f"Retrieved {len(results)} documents")

    # HARD GUARD (important fix)
    if not results or len(results) == 0:
        return {
            "answer": "No relevant regulatory context found to generate a reliable draft.",
            "sources": [],
            "confidence_score": 0.0,
            "warning": "No retrieval results"
        }

    # Optional: filter weak matches
    results = [r for r in results if r.get("similarity_score", 0) > 0.15]

    if len(results) == 0:
        return {
            "answer": "Retrieved documents were not relevant enough to generate a reliable draft.",
            "sources": [],
            "confidence_score": 0.0,
            "warning": "Low relevance retrieval"
        }

    # ---------------------------
    # STEP 2: Build context
    # ---------------------------
    context = "\n\n---\n\n".join(
        f"[Source: {doc.get('metadata', {}).get('source_file', 'unknown')}]\n{doc.get('content', '')}"
        for doc in results
    )

    sources = [
        {
            "source": doc.get("metadata", {}).get("source_file", "unknown"),
            "page": doc.get("metadata", {}).get("page", "unknown"),
            "score": doc.get("similarity_score", 0.0)
        }
        for doc in results
    ]

    confidence_score = (
        max(doc.get("similarity_score", 0.0) for doc in results)
    )

    # ---------------------------
    # STEP 3: Call LLM agent
    # ---------------------------
    try:
        response = drafting_agent.invoke({
            "context": context,
            "input": query
        })

        return {
            "answer": extract_answer(response),
            "sources": sources,
            "confidence_score": confidence_score,
            "warning": (
                "Potential Regulatory Risk Identified"
                if confidence_score < 0.45
                else None
            )
        }

    except Exception as e:
        return {
            "answer": f"Error generating draft: {str(e)}",
            "sources": sources,
            "confidence_score": confidence_score,
            "warning": "Generation failure"
        }

In [ ]:
answer = rag_advanced("What are the required sections of the ICH E3 clinical study report synopsis?",rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What are the required sections of the ICH E3 clinical study report synopsis?'
Top K: 5, Score threshold: 0.0
Metadata Filter: None
Retrieved 5 documents
{'answer': '**Title:** ICH E3 Clinical Study Report Synopsis\n\n**Purpose:** The ICH E3 clinical study report synopsis is intended to provide a concise overview of the clinical study report, highlighting the key information and results.\n\n**Key Regulatory Considerations:**\n\n* The ICH E3 guidance focuses on the report format for interventional clinical trials, but the basic principles can be applied to other types of clinical studies.\n* The design of the study report should be part of the quality by design process.\n* The report should describe the conduct of clinical studies of drug and biological products.\n\n**Draft Content:**\n\n1. **Study Overview**\n\t* Type of study (interventional or observational)\n\t* Information being reported\n2. **Study Objectives**\n\t* Type and objectives of the study\

### Validating Agent

In [ ]:
# ---------------------------
# VALIDATING AGENT 
# ---------------------------
validating_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt= """
                You are a Regulatory Compliance Validation System.

                You evaluate regulatory documents strictly against the provided context derived from:
                - ICH E6(R3)
                - ICH E3
                - ICH E8(R1)
                - FDA Real-World Evidence (RWE) guidance

                ---

                SYSTEM CONSTRAINTS (MANDATORY):
                1. Use ONLY the provided context. Do NOT use external knowledge.
                2. Do NOT assume or infer missing information.
                3. Do NOT generate conversational or explanatory text.
                4. Output must be precise, structured, and machine-readable.
                5. Do NOT modify the document directly.
                6. Every issue must include an actionable fix instruction.
                7. Do NOT include any narrative, reasoning, or explanation outside output format.

                ---

                TASK:
                Evaluate the document for:
                - Structural completeness (ICH E3)
                - Compliance (ICH E6(R3))
                - Study design alignment (ICH E8(R1))
                - FDA RWE usage (if applicable)

                ---

                EVALUATION RULES:
                - Missing section → Violation
                - Incomplete section → Issue
                - Proper section → Compliant
                - If "Information not provided." is used correctly → DO NOT flag as issue

                ---

                DECISION LOGIC:

                IF ANY issues or violations are found:
                → Return structured validation output (see format below)

                IF NO issues or violations:
                → Return EXACT STRING:
                DOCUMENT IS SUBMISSION READY

                ---

                OUTPUT FORMAT (STRICT JSON STYLE):

                {
                "verdict": "PASS" | "FAIL",
                "accuracy_score": "<0-100>",
                "issues": [
                    {
                    "section": "<section name>",
                    "status": "MISSING | INCOMPLETE | VIOLATION",
                    "problem": "<what is wrong>",
                    "fix": "<exact actionable instruction for correction>"
                    }
                ],
                "summary": "brief structured summary of compliance state"
                }

                ---

                INPUTS:

                CONTEXT:
                {context}

                DOCUMENT:
                {draft}

                ---

                OUTPUT RULES:
                - If issues exist → return ONLY JSON object
                - If no issues → return ONLY:
                DOCUMENT IS SUBMISSION READY
        """
             
)

# ---------------------------
# RAG ADVANCED PIPELINE USAGE
# ---------------------------
def validation_pipeline(draft_text, retriever, validating_agent, top_k=5, min_score=0.25):
    """
    Validation RAG Pipeline:
    - Retrieves same context as drafting
    - Compares draft vs documents
    - Produces strict audit report
    """

    print("Running validation pipeline...")

    # ---------------------------
    # STEP 1: USE ADVANCED RAG
    # ---------------------------
    rag_result = rag_advanced(
        query=draft_text,
        retriever=retriever,
        llm = llm,
        top_k=top_k,
        min_score=min_score,
        return_context=True
    )

    context = rag_result.get("context" , "")
    sources_raw = rag_result.get("sources", [])
    confidence_score = rag_result.get("confidence_score" , 0.0)

    # ---------------------------
    # STEP 2: LOW CONTEXT SAFETY
    # ---------------------------
    if confidence_score == 0.0 or not context:
        return {
            "validation_report": "Cannot validate due to insufficient context.",
            "sources": [],
            "confidence_score": 0.0,
            "verdict": "FAIL"
        }

    sources = list(set([doc["source"] for doc in sources_raw]))

    # ---------------------------
# STEP 3: RUN VALIDATION
# ---------------------------
    try:
        response = validating_agent.invoke({
            "context": context,
            "draft": draft_text
        })

        # ---------------------------
        # STEP 4: NORMALIZED OUTPUT HANDLING
        # ---------------------------

        output = None

        # Case 1: dict response (AgentExecutor style)
        if isinstance(response, dict):

            if "messages" in response and response["messages"]:
                output = getattr(response["messages"][-1], "content", str(response["messages"][-1]))

            elif "output" in response:
                output = response["output"]

            elif "content" in response:
                output = response["content"]

            else:
                output = str(response)

        # Case 2: direct LLM / AIMessage
        else:
            output = getattr(response, "content", str(response))

        # ---------------------------
        # OPTIONAL: CLEAN SUCCESS STRING
        # ---------------------------
        if isinstance(output, str) and "DOCUMENT IS SUBMISSION READY" in output:
            output = {
                "verdict": "PASS",
                "summary": "Document is submission ready"
            }

            return {
                "validation_report": output,
                "sources": sources,
                "confidence_score": confidence_score,
                "verdict": "UNKNOWN" 
            }

    except Exception as e:
        return {
            "validation_report": f"Validation error: {str(e)}",
            "sources": sources,
            "confidence_score": confidence_score,
            "verdict": "FAIL"
        }

### Testing Drafting agent only

In [ ]:
query = "Write a CSR synopsis for a Phase 3 oncology study"

draft_result = drafting_pipeline(query, rag_retriever, drafting_agent)

print("DRAFT OUTPUT:\n")
print(draft_result["answer"])
print("\nSOURCES:", draft_result["sources"])
print("CONFIDENCE:", draft_result["confidence_score"])

Processing drafting request: Write a CSR synopsis for a Phase 3 oncology study
Retrieving documents for query: 'Write a CSR synopsis for a Phase 3 oncology study'
Top K: 5, Score threshold: 0.0
Metadata Filter: None
Retrieved 5 documents
Retrieved 5 documents
DRAFT OUTPUT:

**INITIAL DRAFT**

**Title:** Clinical Investigation Report

**Synopsis:** This report summarizes the results of a clinical investigation conducted to evaluate the safety and efficacy of [Insert Product/Device Name].

**Introduction:**
The clinical investigation was conducted in accordance with Good Clinical Practice (GCP) guidelines and regulatory requirements. The objective of this investigation was to assess the safety and efficacy of [Insert Product/Device Name] in [Insert Population/Indication].

**Objectives:**

1. To evaluate the safety of [Insert Product/Device Name] in [Insert Population/Indication].
2. To assess the efficacy of [Insert Product/Device Name] in [Insert Population/Indication].

**Study Design

In [ ]:
print(dir(rag_retriever))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', 'embedding_manager', 'retrieve', 'vector_store']


In [ ]:
docs = rag_retriever.retrieve("oncology trial CSR synopsis")

print(docs)

Retrieving documents for query: 'oncology trial CSR synopsis'
Top K: 5, Score threshold: 0.0
Metadata Filter: None
Retrieved 5 documents
[{'id': 'doc_c5bb913d_714', 'content': 'with the object of ascertaining its safety and/or efficacy.  \n \nClinical trial/Study report (CSR) \n \nA documented description of a trial of any investigational product conducted in human \nparticipants, in which the clinical and statistical description, presentations, and analyses are \nfully integrated into a single report (see the ICH guidance for industry E3 Structure and \nContent of Clinical Study Reports (July 1996)). \n \nComparator  \n \nAn investigational or authorized medicinal product (i.e., active control), placebo or standard \nof care used as a reference in a clinical trial. \n \nCompliance (in relation to trials) \n \nAdherence to the trial-related requirements, GCP requirements, and the applicable regulatory \nrequirements. \n \nConfidentiality', 'metadata': {'producer': 'Adobe PDF Library 25

In [ ]:
draft_result = drafting_pipeline(
    "Phase 3 oncology CSR synopsis template",
    rag_retriever,
    drafting_agent
)

print(draft_result["answer"])

Processing drafting request: Phase 3 oncology CSR synopsis template
Retrieving documents for query: 'Phase 3 oncology CSR synopsis template'
Top K: 5, Score threshold: 0.0
Metadata Filter: None
Retrieved 5 documents
Retrieved 5 documents
Retrieved documents were not relevant enough to generate a reliable draft.


In [ ]:
# ---------------------------
# TEST CASE 1: WEAK CSR (should FAIL)
# ---------------------------
weak_csr = """
Title: Clinical Study Report Synopsis

This study evaluated an oncology drug in cancer patients.

The study was randomized.

The drug showed improvement in outcomes.

Safety was acceptable.

Conclusion: The drug is effective and safe.
"""

print("\n==============================")
print("🧪 TEST CASE 1: WEAK CSR")
print("==============================\n")

result_weak = validation_pipeline(
    draft_text=weak_csr,
    retriever=rag_retriever,
    validating_agent=validating_agent,
    top_k=5,
    min_score=0.1
)

print("📄 VALIDATION REPORT:\n", result_weak["validation_report"])
print("\n📚 SOURCES:", result_weak["sources"])
print("\n📊 CONFIDENCE:", result_weak["confidence_score"])
print("\n🏁 VERDICT:", result_weak.get("verdict"))


# ---------------------------
# TEST CASE 2: STRONG CSR (should PASS or near PASS)
# ---------------------------
strong_csr = """
Title: Clinical Study Report Synopsis

Study Design:
Randomized, double-blind, placebo-controlled Phase 3 trial conducted across multiple sites.

Objectives:
Primary objective was to evaluate progression-free survival. Secondary objectives included overall survival and safety assessment.

Population:
Adult patients with advanced non-small cell lung cancer.

Methodology:
Patients were randomized 1:1 to treatment or placebo. Treatment was administered per protocol-defined schedule.

Results:
The treatment arm demonstrated statistically significant improvement in progression-free survival (HR 0.72, p<0.05).

Safety:
Adverse events were consistent with known safety profile of the investigational drug.

Conclusion:
The study demonstrated clinically meaningful benefit with acceptable safety profile.
"""

print("\n==============================")
print("🧪 TEST CASE 2: STRONG CSR")
print("==============================\n")

result_strong = validation_pipeline(
    draft_text=strong_csr,
    retriever=rag_retriever,
    validating_agent=validating_agent,
    top_k=5,
    min_score=0.1
)

print("📄 VALIDATION REPORT:\n", result_strong["validation_report"])
print("\n📚 SOURCES:", result_strong["sources"])
print("\n📊 CONFIDENCE:", result_strong["confidence_score"])
print("\n🏁 VERDICT:", result_strong.get("verdict"))


# ---------------------------
# TEST CASE 3: EDGE CASE (missing sections)
# ---------------------------
edge_csr = """
Title: Clinical Study Report Synopsis

This study evaluated a treatment in oncology patients.

The study was conducted across multiple centers.

Results showed improvement.

"""

print("\n==============================")
print("🧪 TEST CASE 3: EDGE CASE CSR")
print("==============================\n")

result_edge = validation_pipeline(
    draft_text=edge_csr,
    retriever=rag_retriever,
    validating_agent=validating_agent,
    top_k=5,
    min_score=0.1
)

print("📄 VALIDATION REPORT:\n", result_edge["validation_report"])
print("\n📚 SOURCES:", result_edge["sources"])
print("\n📊 CONFIDENCE:", result_edge["confidence_score"])
print("\n🏁 VERDICT:", result_edge.get("verdict"))


🧪 TEST CASE 1: WEAK CSR

Running validation pipeline...
Retrieving documents for query: '
Title: Clinical Study Report Synopsis

This study evaluated an oncology drug in cancer patients.

The study was randomized.

The drug showed improvement in outcomes.

Safety was acceptable.

Conclusion: The drug is effective and safe.
'
Top K: 5, Score threshold: 0.0
Metadata Filter: None
Retrieved 5 documents
📄 VALIDATION REPORT:
 Cannot validate due to insufficient context.

📚 SOURCES: []

📊 CONFIDENCE: 0.0

🏁 VERDICT: FAIL

🧪 TEST CASE 2: STRONG CSR

Running validation pipeline...
Retrieving documents for query: '
Title: Clinical Study Report Synopsis

Study Design:
Randomized, double-blind, placebo-controlled Phase 3 trial conducted across multiple sites.

Objectives:
Primary objective was to evaluate progression-free survival. Secondary objectives included overall survival and safety assessment.

Population:
Adult patients with advanced non-small cell lung cancer.

Methodology:
Patients were

## Making agent workflows


### Step1 : Generating draft and saving session information

In [ ]:
import uuid


def generate_draft_session(
    user_query,
    retriever,
    drafting_agent
):
    """
    Handles:
    - draft generation
    - config/thread ID creation
    - preparing output for validation pipeline

    Returns:
    {
        "config_id": str,
        "original_query": str,
        "draft_output": str,
        "validation_input": str,
        "sources": list,
        "confidence": float
    }
    """

    print(f"Starting drafting session for: {user_query}")

    # Step 1: Generate unique config/thread ID
    config_id = str(uuid.uuid4())

    # Step 2: Call drafting pipeline
    draft_result = drafting_pipeline(
        query=user_query,
        retriever=retriever,
        drafting_agent=drafting_agent
    )

    draft_output = draft_result["answer"]
    sources = draft_result.get("sources", [])
    confidence = draft_result.get("confidence", 0.0)

    # Step 3: Prepare validation-ready input
    validation_input = f"""
        Please validate the following Clinical Study Report draft
        for ICH E3 compliance, completeness, formatting issues,
        missing sections, and regulatory inconsistencies.

        Original User Request:
        {user_query}

        Generated Draft:
        {draft_output}

        Return only:
        - issues found
        - missing sections
        - correction instructions

        Do NOT rewrite the draft.
        """

    # Step 4: Return structured response
    return {
        "config_id": config_id,
        "original_query": user_query,
        "draft_output": draft_output,
        "validation_input": validation_input,
        "sources": sources,
        "confidence": confidence
    }

### Test Drafting Function Alone

In [ ]:
draft_session = generate_draft_session(
    user_query="Write a CSR synopsis for a Phase 3 oncology study",
    retriever=rag_retriever,
    drafting_agent=drafting_agent
)

print(draft_session)

Starting drafting session for: Write a CSR synopsis for a Phase 3 oncology study
Processing drafting request: Write a CSR synopsis for a Phase 3 oncology study
Retrieving documents for query: 'Write a CSR synopsis for a Phase 3 oncology study'
Top K: 5, Score threshold: 0.0
Metadata Filter: None
Retrieved 5 documents
Retrieved 5 documents
{'config_id': '1345de51-cc5c-42a6-8999-115c2b450b69', 'original_query': 'Write a CSR synopsis for a Phase 3 oncology study', 'draft_output': '**INITIAL DRAFT**\n\n**Title:** Clinical Study Report\n\n**Synopsis:** This clinical study report summarizes the results of a [study type, e.g., Phase 3] clinical trial evaluating the safety and efficacy of [study drug/device].\n\n**Introduction:**\nThe purpose of this clinical study was to assess the safety and efficacy of [study drug/device] in [patient population]. The study was conducted in accordance with the principles of Good Clinical Practice (GCP) and the International Conference on Harmonisation (ICH) 

### Step2: Validate and prepare feedback for drafting agent

In [ ]:
def validate_and_prepare_revision(
    config_id,
    original_query,
    draft_output,
    validation_input,
    validation_agent
):
    """
    Handles:
    - calling validation agent
    - extracting validation feedback
    - preparing revision prompt for drafting agent

    Returns:
    {
        "config_id": str,
        "validation_feedback": str,
        "revision_input": str,
        "status": str
    }
    """

    print(f"Starting validation for config_id: {config_id}")

    # Step 1: Call validation agent
    response = validation_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": validation_input
            }
        ]
    })

    # Step 2: Safe response extraction
    try:
        validation_feedback = response["messages"][-1].content
    except:
        validation_feedback = str(response)

    print("\nVALIDATION FEEDBACK:\n")
    print(validation_feedback)

    # Step 3: Check if revision is needed
    if "no major issues" in validation_feedback.lower():
        return {
            "config_id": config_id,
            "validation_feedback": validation_feedback,
            "revision_input": None,
            "status": "approved"
        }

    # Step 4: Prepare drafting-ready revision prompt
    revision_input = f"""
        Please revise and improve the previous Clinical Study Report draft
        based on the validation feedback below.

        Original User Request:
        {original_query}

        Previous Draft:
        {draft_output}

        Validation Feedback:
        {validation_feedback}

        Instructions:
        - Fix all identified issues
        - Maintain ICH E3 compliance
        - Keep professional regulatory writing style
        - Do not remove valid existing sections
        - Improve completeness and accuracy

        Generate the improved draft now.
        """

    return {
        "config_id": config_id,
        "validation_feedback": validation_feedback,
        "revision_input": revision_input,
        "status": "needs_revision"
    }

### Testing validating function alone

In [ ]:
validation_result = validate_and_prepare_revision(
    config_id=draft_session["config_id"],
    original_query=draft_session["original_query"],
    draft_output=draft_session["draft_output"],
    validation_input=draft_session["validation_input"],
    validation_agent=validating_agent
)

print(validation_result)

Starting validation for config_id: 1345de51-cc5c-42a6-8999-115c2b450b69

VALIDATION FEEDBACK:

**CONTEXT:**
{
  "ICH E3": {
    "required_sections": [
      "Title",
      "Synopsis",
      "Introduction",
      "Objectives",
      "Study Design",
      "Methodology",
      "Results",
      "Discussion",
      "Conclusion",
      "References"
    ],
    "section_requirements": {
      "Synopsis": "Provide a concise summary of the study",
      "Introduction": "Describe the purpose and background of the study",
      "Objectives": "List the primary and secondary objectives of the study",
      "Study Design": "Describe the study design, including the type of study and the number of sites and countries",
      "Methodology": "Describe the methods used to conduct the study, including patient enrollment and randomization",
      "Results": "Present the results of the study, including any data and statistics",
      "Discussion": "Interpret the results of the study and discuss any implicati

### Step3: Revision loop

In [ ]:
def revise_draft_from_validation(
    config_id,
    original_query,
    previous_draft,
    validation_feedback,
    retriever,
    drafting_agent
):
    """
    Takes validation feedback and sends it back to the drafting agent
    to generate an improved draft.

    Returns:
    {
        "config_id": str,
        "improved_draft": str,
        "status": str
    }
    """

    print(f"Revising draft for config_id: {config_id}")

    # Step 1: Prepare revision prompt
    revision_query = f"""
Please improve the previous Clinical Study Report (CSR) draft
based on the validation feedback below.

Original User Request:
{original_query}

Previous Draft:
{previous_draft}

Validation Feedback:
{validation_feedback}

Instructions:
- Fix all identified issues
- Maintain ICH E3 compliance
- Keep professional regulatory writing style
- Do not remove valid existing sections
- Improve completeness and accuracy

Generate the improved final draft.
"""

    # Step 2: Send to drafting pipeline
    revised_result = drafting_pipeline(
        query=revision_query,
        retriever=retriever,
        drafting_agent=drafting_agent
    )

    improved_draft = revised_result["answer"]

    return {
        "config_id": config_id,
        "improved_draft": improved_draft,
        "status": "revised"
    }

### Pdf Generation

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4


def export_to_pdf(config_id, improved_draft):
    """
    Converts final improved draft into PDF

    Returns:
    PDF file path
    """

    file_name = f"CSR_Report_{config_id}.pdf"

    doc = SimpleDocTemplate(
        file_name,
        pagesize=A4
    )

    styles = getSampleStyleSheet()
    content = []

    # Title
    title = Paragraph(
        "Clinical Study Report (CSR) - Final Draft",
        styles["Title"]
    )
    content.append(title)
    content.append(Spacer(1, 20))

    # Split long text into paragraphs
    paragraphs = improved_draft.split("\n")

    for para in paragraphs:
        if para.strip():
            content.append(
                Paragraph(para, styles["Normal"])
            )
            content.append(Spacer(1, 10))

    # Build PDF
    doc.build(content)

    print(f"PDF generated successfully: {file_name}")

    return file_name

### Testing the revision 

In [ ]:
final_revision = revise_draft_from_validation(
    config_id=draft_session["config_id"],
    original_query=draft_session["original_query"],
    previous_draft=draft_session["draft_output"],
    validation_feedback=validation_result["validation_feedback"],
    retriever=rag_retriever,
    drafting_agent=drafting_agent
)

print(final_revision)

Revising draft for config_id: 1345de51-cc5c-42a6-8999-115c2b450b69
Processing drafting request: 
Please improve the previous Clinical Study Report (CSR) draft
based on the validation feedback below.

Original User Request:
Write a CSR synopsis for a Phase 3 oncology study

Previous Draft:
**INITIAL DRAFT**

**Title:** Clinical Study Report

**Synopsis:** This clinical study report summarizes the results of a [study type, e.g., Phase 3] clinical trial evaluating the safety and efficacy of [study drug/device].

**Introduction:**
The purpose of this clinical study was to assess the safety and efficacy of [study drug/device] in [patient population]. The study was conducted in accordance with the principles of Good Clinical Practice (GCP) and the International Conference on Harmonisation (ICH) guidelines.

**Objectives:**

1. To evaluate the efficacy of [study drug/device] in [primary endpoint].
2. To assess the safety of [study drug/device] in [patient population].

**Study Design:**
This 

In [ ]:
import re

def clean_markdown(text):
    text = re.sub(r'#+\s*', '', text)          # remove headings
    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)  # bold
    text = re.sub(r'\*(.*?)\*', r'\1', text)      # italic
    text = re.sub(r'`(.*?)`', r'\1', text)        # code
    text = re.sub(r'-\s+', '', text)              # bullets
    text = re.sub(r'\n{3,}', '\n\n', text)        # extra newlines
    return text.strip()


clean_text = clean_markdown(final_revision["improved_draft"])

pdf_file = export_to_pdf(
    config_id=final_revision["config_id"],
    improved_draft=clean_text
)

PDF generated successfully: CSR_Report_1345de51-cc5c-42a6-8999-115c2b450b69.pdf


In [ ]:
from IPython.display import FileLink

FileLink(pdf_file)

c:\Users\shrut\Cognizant Technoverse\Correct\Backend\CSR_Report_1345de51-cc5c-42a6-8999-115c2b450b69.pdf